In [1]:
import requests
import math
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import json

load_dotenv()

KEY = os.getenv('API_KEY')
LIST_URL = "https://play.limitlesstcg.com/api/tournaments/"
URL = LIST_URL+"{}"
FILENAME = "data.xlsx"

# Tournaments List

In [2]:
start_date = datetime(2026, 8, 31).date()
end_date = datetime(2026, 9, 13).date()

In [3]:
response = requests.get(LIST_URL, headers={'X-Access-Key':KEY}, params={'game': 'PTCG', 'format': 'STANDARD', 'limit': 1000})

In [4]:
tournament_list = []
id_list = []
date_format = "%Y-%m-%dT%H:%M:%S.%fZ"
i = 0
for entry in response.json():
    # print(entry)
    date = (datetime.strptime(entry['date'], date_format) - timedelta(hours=5)).date()
    if entry['players'] >= 60 and date >= start_date and date <= end_date:
        print(entry['id'], entry['name'] )
        tournament_list.append('{}_{}'.format(i,entry['name']))
        id_list.append(entry['id'])
        i+=1


6aa38dfaab080c8c95800d97 Rare Candy Club Showdown #44 (50 CODES)
6aa0a22dab080c8c957fe961 #Triplo Pack Brazil Limitless Trainer  EleuCards #
6aa6f2dd38886b36383bf003 The Dark League | Mefford Birthday Tournament
6aa2e21aab080c8c958002e4 SEASAC League Challenge #6 (SEASON 5) [50 CODES]
6a9ceedbab080c8c957fb8f2 Vault Of Games #2🏆|SEASON 1|WEEKLY
6a63ce2252c24ac2da64ae00 RDG! September Open Qualifier
6a99a15fa4272c53be645cd9 💲100 Cash PTCGL Welcome To The HIDEOUT
6aa2e234a4272c53be64d0b0 SEASAC League Cup #1 (SEASON 5) [150 CODES]
6a87214678baaa6d1c2244db TOURNAMENT OF DOOM! SPARKLING WATER!
6aa127fba4272c53be64b8e6 ASRcristiano # 187 - 60 codes - Road to Frankfurt!
6a9eece8a4272c53be649d53 ⚡ Surge's TCG Vault Winner Takes All Live BO3 #22
6aa18cb7a4272c53be64bcbb Team Corna Weekly September's #3
6aa3c275f1243e65f97fce85 The Dark League | Baltimore Regionals Testing
6a438738c87cd9ff0a31356f Oceania Open! | Season 3 - Week 11
6aa0dc1aa4272c53be64b7be 🟢Jolly Good Weekly #6 | Top Cut! (50 Co

# Tournament Data

In [5]:
deck_df = pd.DataFrame(columns=['Player', 'Nation', 'Deck', 'Tournament', 'Placement', 'Day2'])
# standing_df = pd.DataFrame(columns=['Player', 'Wins', 'Losses', 'Ties'])
# pairings_df = pd.DataFrame(columns=['Tour', 'Round', 'Player', 'Opponent', 'Result'])
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])

with open('archetype.json', 'r') as file:
    arch_dict = json.load(file)
    # deck_df['Deck'] = deck_df['Deck'].apply(lambda x: arch_dict[x] if x in arch_dict else x)

deck_dict = deck_df.set_index('Player')['Deck'].to_dict()

In [6]:
def json_to_decklist(decklist_json):
    decklist = ""
    for key in decklist_json.keys():
        for card in decklist_json[key]:
            card_count = card['count']
            card_name = card['name']
            card_set = card['set']
            card_number = card['number']
            line = f"{card_count} {card_name} {card_set} {card_number}\n"
            decklist += line
    return decklist


In [7]:
def variant_classification(deck, decklist, variant_dict):
    if deck in variant_dict:
        for deck_key in variant_dict[deck]:
            for cat_key in variant_dict[deck][deck_key]:
                    if all(any(card["name"] == card_key and card["count"] >= variant_dict[deck][deck_key][cat_key][card_key] for card in decklist[cat_key]) for card_key in variant_dict[deck][deck_key][cat_key]):
                        return deck_key
    return deck

In [8]:
FILENAME = "data.xlsx"

other_count = 0
# standings = requests.get(URL.format(id_list[0])+"/standings", headers={'X-Access-Key':KEY})

deck_df = pd.DataFrame(columns=['Player', 'Nation', 'Deck', 'Tournament', 'Placement', 'Day2'])
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])
# with open('variant_class.json', 'r') as file:
#     variant_dict = json.load(file)

# with open('archetype.json', 'r') as file:
#     arch_dict = json.load(file)
#     deck_df['Deck'] = deck_df['Deck'].apply(lambda x: arch_dict[x] if x in arch_dict else x)

deck_dict = deck_df.set_index('Player')['Deck'].to_dict()

ignore_list = ["6995493ec9dc8186f760750a", "6999e718c9dc8186f7609712", "699d3b523ef65252b05758b0", "699f90cfc9dc8186f760c374"]

for id, tour in zip(id_list, tournament_list):
    if id in ignore_list:
        continue
    standings = requests.get(URL.format(id)+"/standings", headers={'X-Access-Key':KEY})
    pairings = requests.get(URL.format(id)+"/pairings", headers={'X-Access-Key':KEY})
    phases = requests.get(URL.format(id)+"/details", headers={'X-Access-Key':KEY}).json()['phases']
    if not all('name' in entry['deck'] for entry in standings.json()):
        print("SKIPPED:", id, tour)
        continue

    # print(tour)
    top_size = 0
    swiss_rounds = 0
    if len(phases) <= 1:
        top_size = 8
    else:
        for phase in phases:
            if phase['type'] != 'SWISS': break
            swiss_rounds += phase['rounds']
    # i = 1
    for entry in standings.json():

        # if entry['placing'] == 'None': continue
        name = "{}_{}".format(tour, entry['player'])
        nation = entry['country']
        wins = entry['record']['wins']
        losses = entry['record']['losses']
        ties = entry['record']['ties']
        deck = arch_dict[entry['deck']['name']] if entry['deck']['name'] in arch_dict else entry['deck']['name']

        # if 'decklist' in entry:
        #     deck = variant_classification(deck, entry['decklist'], variant_dict)
        
        if deck == "Other":
            other_count += 1
        # deck_df.loc[len(deck_df)] = name, entry['deck']['name']

        if top_size > 0:
            if entry['placing'] != None and entry['placing'] < (top_size+1):
                deck_df.loc[len(deck_df)] = name, nation, deck, tour, "Top {}".format(pow(2, math.ceil(math.log(entry['placing'], 2)))), True
            else:
                deck_df.loc[len(deck_df)] = name, nation, deck, tour, "Out of Top", False
        else:
            if entry['placing'] != None and (wins + losses + ties) > swiss_rounds:
                deck_df.loc[len(deck_df)] = name, nation, deck, tour, "Top {}".format(pow(2, math.ceil(math.log(entry['placing'], 2)))), True
            else:
                deck_df.loc[len(deck_df)] = name, nation, deck, tour, "Out of Top", False

    for entry in pairings.json():
        # if entry['round'] < 4:
        #     continue
        try:
            player = "{}_{}".format(tour, entry['player1'])
            opponent = "{}_{}".format(tour, entry['player2'])
            player_deck = deck_df.loc[deck_df['Player'] == player]['Deck'].values[0]
            opp_deck = deck_df.loc[deck_df['Player'] == opponent]['Deck'].values[0]
            matchup = matchups_df.loc[(matchups_df['Deck'] == player_deck) & (matchups_df['Opposing Deck'] == opp_deck)]
            inv_matchup = matchups_df.loc[(matchups_df['Deck'] == opp_deck) & (matchups_df['Opposing Deck'] == player_deck)]
            if entry['winner'] == 0:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'T'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'T'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 1
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 0, 0, 1
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 2
                else:
                    matchups_df.loc[matchup.index, 'Ties'] += 1
                    matchups_df.loc[inv_matchup.index, 'Ties'] += 1
            elif entry['player1'] == entry['winner']:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'W'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'L'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 0, 0
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 0, 1, 0
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 1, 0
                else:
                    matchups_df.loc[matchup.index, 'Wins'] += 1
                    matchups_df.loc[inv_matchup.index, 'Losses'] += 1
            elif entry['player2'] == entry['winner']:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'L'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'W'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 1, 0
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 1, 0, 0
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 1, 0
                else:
                    matchups_df.loc[matchup.index, 'Losses'] += 1
                    matchups_df.loc[inv_matchup.index, 'Wins'] += 1
        except:
            continue




In [9]:
other_count

261

In [10]:
with pd.ExcelWriter(FILENAME) as writer:
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    # standing_df.to_excel(writer, sheet_name='standings', index=False)
    # pairings_df.to_excel(writer, sheet_name='pairings', index=False)
    matchups_df.to_excel(writer, sheet_name='matchups', index=False)